# EchoPilot Research 05: End-to-End Local Pipeline Benchmark

This notebook tests the entire offline pipeline end-to-end:
1. **Document Ingestion**: Parsing PDF and creating page-aware chunks.
2. **Material-Isolated Indexing**: Embedding chunks with `all-MiniLM-L6-v2`.
3. **Adaptive Skill Selection**: Querying BKT states to find the weakest skill.
4. **Passage Retrieval**: Fetching top-K supporting evidence chunks.
5. **Question Generation**: Creating a 5-question multiple choice quiz.
6. **Schema & Grounding Verification**: Enforcing deterministic checks.
7. **Simulation of Student Quiz**: Submitting answers and running BKT updates.


In [ ]:
import os
import sys
import json

sys.path.append(os.path.abspath("../src"))
from chunker import chunk_document
from retriever import MaterialVectorStore
from generator import build_quiz_generation_prompt, validate_and_parse_quiz_json
from bkt import BKTModel

print("All EchoPilot pipeline modules imported.")


## Step 1: Simulate Document Ingestion
We index a study material and initialize its per-skill BKT states.


In [ ]:
sample_pages = [
    {"page_number": 1, "text": "Operating Systems: Deadlock Overview. Deadlock is a state where processes are blocked waiting for resources held by each other. The 4 conditions are Mutual Exclusion, Hold and Wait, No Preemption, and Circular Wait."},
    {"page_number": 2, "text": "Deadlock Avoidance: Banker's Algorithm uses Available, Max, Allocation, and Need matrices. Need[i, j] = Max[i, j] - Allocation[i, j]."},
    {"page_number": 3, "text": "Resource Allocation Graph: In single-instance resource graphs, cycle detection confirms deadlock. For multi-instance graphs, a cycle is necessary but not sufficient."}
]

chunks = chunk_document(sample_pages, chunk_size=100, chunk_overlap=20, material_id="mat_demo_101", material_name="Deadlock_Lecture.pdf")
store = MaterialVectorStore("sentence-transformers/all-MiniLM-L6-v2")
store.index_chunks(chunks)

print(f"Indexed {len(chunks)} chunks for 'Deadlock_Lecture.pdf'.")


## Step 2: Adaptive Skill Selection
Suppose the learner has existing BKT mastery states:
- 'Deadlock Overview': $P(L) = 0.82$
- 'Resource Allocation Graph': $P(L) = 0.65$
- 'Banker Algorithm': $P(L) = 0.28$ (Weakest skill)


In [ ]:
learner_bkt = {
    "Deadlock Overview": {"mastery": 0.82, "model": BKTModel(p_l0=0.3, p_t=0.15, p_g=0.2, p_s=0.1)},
    "Resource Allocation Graph": {"mastery": 0.65, "model": BKTModel(p_l0=0.25, p_t=0.12, p_g=0.2, p_s=0.1)},
    "Banker Algorithm": {"mastery": 0.28, "model": BKTModel(p_l0=0.15, p_t=0.10, p_g=0.25, p_s=0.12)}
}

# Select weakest topic
weakest_skill = min(learner_bkt.items(), key=lambda x: x[1]["mastery"])
target_topic = weakest_skill[0]
current_mastery = weakest_skill[1]["mastery"]

print(f"Selected Practice Topic: '{target_topic}' (Current Mastery: {current_mastery:.1%})")
difficulty = "gentle" if current_mastery < 0.40 else "steady"
print(f"Adaptive Target Difficulty: {difficulty}")


## Step 3: Retrieve Grounded Evidence Chunks
Retrieve top passages isolated to this material.


In [ ]:
retrieved = store.retrieve(query=target_topic, top_k=2, filter_material_id="mat_demo_101")
print(f"Retrieved {len(retrieved)} supporting evidence chunks:")
for r in retrieved:
    print(f" - Page {r['page_number']}: {r['text']}")


## Step 4: Build Generation Prompt
Construct the prompt enforcing strict JSON output.


In [ ]:
prompt = build_quiz_generation_prompt(
    material_name="Deadlock_Lecture.pdf",
    topic=target_topic,
    difficulty=difficulty,
    evidence_chunks=retrieved,
    num_questions=3
)
print("Generated Instruction Prompt:")
print(prompt[:300] + "...")


## Step 5: Validate Output & Simulate Student Submission
We parse the structured quiz, simulate a student getting 2 out of 3 questions right, and update the BKT state.


In [ ]:
mock_generated_json = {
    "title": "Banker's Algorithm Check-in",
    "questions": [
        {
            "topic": target_topic,
            "question": "What formula defines the Need matrix in Banker's algorithm?",
            "options": ["Need = Max - Allocation", "Need = Max + Allocation", "Need = Available - Max", "Need = Allocation / Max"],
            "answer": 0,
            "explanation": "Need is defined as Max minus Allocation.",
            "difficulty": "gentle"
        },
        {
            "topic": target_topic,
            "question": "Which data structure tracks available instances of each resource type?",
            "options": ["Allocation", "Available", "Need", "Max"],
            "answer": 1,
            "explanation": "Available is a 1-D array indicating the count of free instances.",
            "difficulty": "gentle"
        },
        {
            "topic": target_topic,
            "question": "What does a safe state guarantee in deadlock avoidance?",
            "options": ["The system can allocate resources to each process in some sequence without deadlock", "All processes run in parallel with zero memory", "Processes never request more than 1 resource", "No process ever terminates"],
            "answer": 0,
            "explanation": "A safe state guarantees at least one deadlock-free execution sequence.",
            "difficulty": "gentle"
        }
    ]
}

parsed_quiz = validate_and_parse_quiz_json(json.dumps(mock_generated_json))
print(f"Validated Quiz: '{parsed_quiz['title']}' with {len(parsed_quiz['questions'])} questions.")

# Student answers: Q1=Correct (0), Q2=Correct (1), Q3=Incorrect (1 instead of 0)
student_answers = [0, 1, 1]
bkt_engine = learner_bkt[target_topic]["model"]
updated_m = current_mastery

print(f"\nInitial Mastery on '{target_topic}': {updated_m:.1%}")
for i, q in enumerate(parsed_quiz["questions"]):
    is_correct = (student_answers[i] == q["answer"])
    updated_m = bkt_engine.update_mastery(updated_m, is_correct)
    print(f" Q{i+1}: {'Correct' if is_correct else 'Incorrect'} -> Updated Mastery: {updated_m:.1%}")

learner_bkt[target_topic]["mastery"] = updated_m
print(f"\nFinal Mastery on '{target_topic}': {updated_m:.1%}")
